# PropRAG-style Two-Stage Graph Retrieval (LLM-free)

Answers the 811 HotpotQA-style bridge queries against the proposition graph built in
`construct_graph.ipynb` (`bridge_gold_graph/graph.gpickle`). No LLM calls anywhere in this
pipeline — retrieval is embeddings (BGE-M3, dense-only) + personalized PageRank + beam search.

Pipeline:
1. **Setup** — load the graph + OpenIE docs, embed every proposition once, cache to disk.
2. **Stage 1 (coarse)** — embed the query, find its top-N similar propositions, seed personalized
   PageRank over the whole graph from their entities, take the top-K passages by PPR score, and
   induce a local subgraph around them.
3. **Stage 2 (beam search)** — pure path expansion + embedding scoring inside that subgraph, no
   PPR involved.
4. **Final scoring** — turn the surviving beam paths into entity scores, combine with the
   stage-1 seed, run a second (subgraph-local) PPR, and rank passages by that.
5. **Batch runner** — loop over all 811 queries, checkpointing to JSONL so re-running is resumable.
6. **Evaluation** — recall@K against `supporting_facts` gold titles, as a sanity check.

## 0. Config

In [ ]:
import json
import pickle
import time
import itertools
import collections
from pathlib import Path

import numpy as np
import networkx as nx
from tqdm.auto import tqdm

# --- Paths -------------------------------------------------------------------
GRAPH_PATH = "bridge_gold_graph/graph.gpickle"
OPENIE_PATH = "openie_results_ner_meta-llama_llama-3.3-70b-instruct.json"
QUERIES_PATH = "hotpotqa.json"

OUTPUT_DIR = Path("bridge_gold_graph")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROP_EMBEDDINGS_PATH = OUTPUT_DIR / "prop_embeddings.npy"
PROP_META_PATH = OUTPUT_DIR / "prop_meta.json"

RESULTS_PATH = Path("retrieval_results.jsonl")
FAILURES_PATH = Path("retrieval_failures.jsonl")

# --- Embedding model -----------------------------------------------------------
# TODO: point this at your local BGE-M3 checkpoint (local path or HF hub id).
BGE_M3_MODEL_PATH = "<path-to-your-BGE-M3-checkpoint>"
BGE_M3_USE_FP16 = True
BGE_M3_BATCH_SIZE = 32
# BGE-M3 does not require a query-side instruction prefix the way e.g. bge-large-en does --
# leave empty unless your checkpoint's model card says otherwise.
BGE_M3_QUERY_INSTRUCTION = ""

# --- Query scope -----------------------------------------------------------------
QUESTION_TYPE = "bridge"     # same scoping rule used to build the graph in construct_graph.ipynb
EXPECTED_NUM_QUERIES = 811   # sanity-checked against the actual filtered count below before batch running

# --- Stage 1: coarse retrieval -----------------------------------------------------
N_PROP_STAGE1 = 20          # top-N propositions by query similarity, used to seed PPR
TOP_K_PASSAGES = 50         # top-K passage nodes by PPR score, kept for the beam-search subgraph
PPR_ALPHA_STAGE1 = 0.75
SUBGRAPH_EXPANSION_HOPS = 2  # passages -> containment-connected entities (hop 1, always) ->
                              # further hyperedge/synonymy entity-entity hops (hops - 1 of them)

# --- Stage 2: beam search --------------------------------------------------------
BEAM_WIDTH = 3
MAX_BEAM_HOPS = 3
NUM_JUMP_POINTS = 3          # top query-similarity propositions injected at every hop regardless
                              # of graph connectivity, guarding against broken/missing graph links

# --- Stage 2 -> entity scoring ----------------------------------------------------
SYNONYMY_BRIDGE_BOOST = 0.5  # extra additive entity_scores boost when consecutive path
                              # propositions are linked via a synonymy edge

# --- Final PPR ---------------------------------------------------------------------
PPR_ALPHA_STAGE2 = 0.45
TOP_K_RESULTS = 10           # number of passages returned per query

# --- Batch runner --------------------------------------------------------------------
PROGRESS_PRINT_AFTER = 20    # print a running avg time/query + ETA after this many queries

# --- Evaluation ------------------------------------------------------------------------
RECALL_AT_K_VALUES = [2, 5, 10]

## 1. Load the graph, OpenIE docs, and queries

Filters `hotpotqa.json` down to `QUESTION_TYPE` and asserts the count matches
`EXPECTED_NUM_QUERIES` (811) before anything downstream runs, per the brief -- if this assertion
fails, the scoping rule here no longer matches whatever produced the 811-query set and needs
adjusting.

In [ ]:
with open(GRAPH_PATH, "rb") as f:
    G = pickle.load(f)

with open(OPENIE_PATH) as f:
    openie_docs = json.load(f)["docs"]

with open(QUERIES_PATH) as f:
    all_queries = json.load(f)


def doc_title(doc):
    # Each passage is formatted as "<Title>\n<body...>" by the OpenIE indexing step.
    return doc["passage"].split("\n", 1)[0]


graph_pids = {n for n, d in G.nodes(data=True) if d.get("node_type") == "passage"}
docs_by_pid = {d["idx"]: d for d in openie_docs if d["idx"] in graph_pids}

print(f"{G.number_of_nodes()} nodes / {G.number_of_edges()} edges loaded from {GRAPH_PATH}")
print(f"{len(graph_pids)} passage nodes in graph, {len(docs_by_pid)} matched against OpenIE docs")

In [ ]:
scoped_queries = [q for q in all_queries if q.get("type") == QUESTION_TYPE]

print(f"{len(all_queries)} total queries in {QUERIES_PATH}")
print(f"{len(scoped_queries)} queries of type '{QUESTION_TYPE}'")

assert len(scoped_queries) == EXPECTED_NUM_QUERIES, (
    f"Expected {EXPECTED_NUM_QUERIES} queries scoped to type='{QUESTION_TYPE}', got "
    f"{len(scoped_queries)} instead -- adjust QUESTION_TYPE / EXPECTED_NUM_QUERIES, or the "
    f"filter in the cell above, to match whatever produced the 811-query set."
)

queries = scoped_queries


def get_query_id(item):
    return item.get("_id") or item.get("id") or item.get("qid")


print(f"Using {len(queries)} queries for retrieval.")

## 2. Embedding model (BGE-M3, dense-only)

Dense embeddings only -- BGE-M3 also supports sparse and multi-vector (ColBERT-style) scoring,
but this pipeline uses neither. `embed_texts` batches the encode calls itself (so we get a real
progress bar and control over `BGE_M3_BATCH_SIZE`) and explicitly L2-normalizes the output, so
`np.dot` downstream is a correct cosine similarity regardless of what the checkpoint already
guarantees.

In [ ]:
from FlagEmbedding import BGEM3FlagModel

embed_model = BGEM3FlagModel(BGE_M3_MODEL_PATH, use_fp16=BGE_M3_USE_FP16)


def embed_texts(texts, batch_size=BGE_M3_BATCH_SIZE, is_query=False, show_progress_bar=False):
    """Dense-only BGE-M3 embeddings, L2-normalized for cosine similarity via dot product."""
    if is_query and BGE_M3_QUERY_INSTRUCTION:
        texts = [BGE_M3_QUERY_INSTRUCTION + t for t in texts]

    chunks = [texts[i:i + batch_size] for i in range(0, len(texts), batch_size)]
    iterator = tqdm(chunks, desc="embedding") if show_progress_bar else chunks

    all_dense = []
    for chunk in iterator:
        output = embed_model.encode(
            chunk,
            batch_size=len(chunk),
            max_length=8192,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )
        all_dense.append(np.asarray(output["dense_vecs"], dtype=np.float32))
    dense = np.concatenate(all_dense, axis=0) if all_dense else np.zeros((0, 1024), dtype=np.float32)

    norms = np.linalg.norm(dense, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return dense / norms


print("BGE-M3 loaded:", BGE_M3_MODEL_PATH)

## 3. Proposition index

Flatten every proposition across all in-scope passages, embed once, and cache to disk
(`prop_embeddings.npy` + `prop_meta.json`) so re-running the notebook doesn't re-embed from
scratch. Also build `entity_to_props` (inverted index, used by beam search) and `prop_entities`.

In [ ]:
def build_propositions(docs_by_pid):
    props = []
    for pid, doc in docs_by_pid.items():
        for i, prop in enumerate(doc["propositions"]):
            props.append({
                "prop_id": f"{pid}::{i}",
                "pid": pid,
                "entities": prop.get("entities", []),
                "text": prop["text"],
            })
    return props


propositions = build_propositions(docs_by_pid)
print(f"{len(propositions)} propositions across {len(docs_by_pid)} passages")

In [ ]:
if PROP_EMBEDDINGS_PATH.exists() and PROP_META_PATH.exists():
    prop_embeddings = np.load(PROP_EMBEDDINGS_PATH)
    with open(PROP_META_PATH) as f:
        prop_meta = json.load(f)
    assert len(prop_meta) == len(propositions) == prop_embeddings.shape[0], (
        f"Cached prop_meta/prop_embeddings ({len(prop_meta)} / {prop_embeddings.shape[0]}) don't "
        f"match the propositions rebuilt from docs_by_pid ({len(propositions)}) -- the cache is "
        f"stale (graph/docs changed since it was written). Delete {PROP_EMBEDDINGS_PATH} and "
        f"{PROP_META_PATH} and re-run this cell to rebuild."
    )
    propositions = prop_meta
    print(f"Loaded cached embeddings: {prop_embeddings.shape} from {PROP_EMBEDDINGS_PATH}")
else:
    texts = [p["text"] for p in propositions]
    prop_embeddings = embed_texts(texts, show_progress_bar=True)
    prop_meta = propositions
    np.save(PROP_EMBEDDINGS_PATH, prop_embeddings)
    with open(PROP_META_PATH, "w") as f:
        json.dump(prop_meta, f)
    print(f"Encoded and cached {prop_embeddings.shape} embeddings to {PROP_EMBEDDINGS_PATH}")

In [ ]:
prop_ids = [p["prop_id"] for p in propositions]
prop_id_to_idx = {pid_: i for i, pid_ in enumerate(prop_ids)}
prop_entities = {p["prop_id"]: p["entities"] for p in propositions}

entity_to_props = collections.defaultdict(list)
for p in propositions:
    for e in p["entities"]:
        entity_to_props[e].append(p["prop_id"])

print(f"{len(entity_to_props)} unique entities indexed across propositions")
print(f"{len(prop_entities)} propositions indexed, embedding dim = {prop_embeddings.shape[1]}")

## 4. Combined edge weight for PageRank

`nx.pagerank`'s `weight` param takes a single attribute name, but graph edges carry up to three
separate weight attributes (`hyperedge_weight`, `synonymy_weight`, `containment_weight`). Compute
a single `weight` = sum of whichever are present (default 0.0), once, up front.

In [ ]:
WEIGHT_COMPONENTS = ["hyperedge_weight", "synonymy_weight", "containment_weight"]

for _, _, data in G.edges(data=True):
    data["weight"] = sum(data.get(k, 0.0) for k in WEIGHT_COMPONENTS)

edge_weights = [d["weight"] for _, _, d in G.edges(data=True)]
print(f"weight attr set on {len(edge_weights)} edges "
      f"(min={min(edge_weights):.2f}, max={max(edge_weights):.2f}, "
      f"mean={sum(edge_weights) / len(edge_weights):.3f})")

## 5. Stage 1 — coarse retrieval

Embed the query, find its top-`N_PROP_STAGE1` propositions, seed personalized PageRank over the
*whole* graph from their entities, take the top-`K` passages by PPR score, and induce a local
subgraph around them (passages -> containment-connected entities -> `SUBGRAPH_EXPANSION_HOPS - 1`
further hyperedge/synonymy entity-entity hops).

In [ ]:
def entity_node_id(entity_str):
    return f"entity::{entity_str}"


def top_propositions_by_similarity(query_vec, n):
    sims = prop_embeddings @ query_vec
    order = np.argsort(-sims)[:n]
    return [(prop_ids[i], float(sims[i])) for i in order]


def build_seed_dict(top_props):
    """Sinitial: entity_node_id -> weight, summing similarity-score contributions across every
    top-N proposition an entity appears in."""
    seed = collections.defaultdict(float)
    for prop_id, score in top_props:
        for ent in prop_entities.get(prop_id, []):
            seed[entity_node_id(ent)] += max(score, 0.0)
    return seed


def normalized_personalization(weighted_nodes, graph):
    """Sparse {node: probability} dict for nx.pagerank's personalization param -- nodes not in
    the graph or with non-positive weight are dropped; the rest are renormalized to sum to 1.
    Returns None (uniform fallback) if nothing survives."""
    filtered = {n: w for n, w in weighted_nodes.items() if n in graph and w > 0}
    total = sum(filtered.values())
    if total <= 0:
        return None
    return {n: w / total for n, w in filtered.items()}


def induce_subgraph(graph, top_passage_ids, hops=SUBGRAPH_EXPANSION_HOPS):
    """passages -> containment-connected entities (hop 1, always) -> that entity set expanded
    outward along entity-entity (hyperedge/synonymy) edges for `hops - 1` further hops."""
    sub_nodes = set(top_passage_ids)

    entity_frontier = set()
    for pid in top_passage_ids:
        for nbr in graph.neighbors(pid):
            if graph.nodes[nbr].get("node_type") == "entity":
                entity_frontier.add(nbr)
    sub_nodes.update(entity_frontier)

    for _ in range(max(hops - 1, 0)):
        next_frontier = set()
        for ent in entity_frontier:
            for nbr in graph.neighbors(ent):
                if graph.nodes[nbr].get("node_type") == "entity":
                    next_frontier.add(nbr)
        sub_nodes.update(next_frontier)
        entity_frontier = next_frontier

    return graph.subgraph(sub_nodes).copy()


def stage1_coarse_retrieval(query_vec):
    top_props = top_propositions_by_similarity(query_vec, N_PROP_STAGE1)
    s_initial = build_seed_dict(top_props)

    personalization = normalized_personalization(s_initial, G)
    ppr_scores = nx.pagerank(G, alpha=PPR_ALPHA_STAGE1, personalization=personalization, weight="weight")

    passage_scores = [
        (n, s) for n, s in ppr_scores.items() if G.nodes[n].get("node_type") == "passage"
    ]
    passage_scores.sort(key=lambda kv: -kv[1])
    top_passage_ids = [pid for pid, _ in passage_scores[:TOP_K_PASSAGES]]

    G_sub = induce_subgraph(G, top_passage_ids)
    return top_props, s_initial, top_passage_ids, G_sub


print("Stage 1 functions defined.")

## 6. Stage 2 — beam search over `G_sub`

No PPR here, pure path expansion + embedding scoring. Seed the beam with the top `beam_width`
propositions (by query similarity) whose entities are present in `G_sub`; at each hop, expand via
`entity_to_props` (graph-connected candidates, restricted to entities present in `G_sub`) plus a
fixed set of top-similarity "jump points" that ignore graph connectivity entirely, guarding
against missing/broken graph links.

In [ ]:
def path_embedding(path):
    idxs = [prop_id_to_idx[p] for p in path]
    return prop_embeddings[idxs].mean(axis=0)


def score_path(path, query_vec):
    emb = path_embedding(path)
    norm = np.linalg.norm(emb)
    if norm == 0:
        return 0.0
    return float(np.dot(emb / norm, query_vec))


def stage2_beam_search(query_vec, sims_by_prop_id, G_sub):
    sub_entity_strs = {d["text"] for _, d in G_sub.nodes(data=True) if d.get("node_type") == "entity"}

    ranked_props = sorted(sims_by_prop_id.items(), key=lambda kv: -kv[1])

    seed_candidates = [
        (pid, score) for pid, score in ranked_props
        if any(e in sub_entity_strs for e in prop_entities.get(pid, []))
    ][:BEAM_WIDTH]
    if not seed_candidates:
        # G_sub had no overlap with any proposition's entities (shouldn't normally happen since
        # G_sub is built from stage 1's own top-N propositions' entities) -- fall back to raw
        # top-similarity propositions so beam search always has somewhere to start.
        seed_candidates = ranked_props[:BEAM_WIDTH]

    beam = [([pid], score) for pid, score in seed_candidates]
    jump_points = [pid for pid, _ in ranked_props[:NUM_JUMP_POINTS]]

    for _ in range(MAX_BEAM_HOPS):
        candidates = []
        for path, _ in beam:
            last_prop = path[-1]
            last_entities = [e for e in prop_entities.get(last_prop, []) if e in sub_entity_strs]

            extension_candidates = set()
            for e in last_entities:
                for cand_prop in entity_to_props.get(e, []):
                    if cand_prop not in path:
                        extension_candidates.add(cand_prop)
            for jp in jump_points:
                if jp not in path:
                    extension_candidates.add(jp)

            for cand_prop in extension_candidates:
                new_path = path + [cand_prop]
                candidates.append((new_path, score_path(new_path, query_vec)))

        if not candidates:
            break  # no further extensions possible from any path currently in the beam

        candidates.sort(key=lambda pc: -pc[1])
        beam = candidates[:BEAM_WIDTH]

    return beam


print("Stage 2 functions defined.")

## 7. Entity scoring from beam paths + final PPR

Every entity in every proposition on a surviving path accumulates `path_score` into
`entity_scores`, with an extra additive `SYNONYMY_BRIDGE_BOOST` when two consecutive propositions
on the path are linked by a synonymy edge (mirrors bridge-entity importance). `entity_scores` and
`s_initial` are combined by simple sum into `Sfinal`, kept as two separately named dicts up to
that point so the combination rule stays easy to see/tune. A second, subgraph-local PPR then
ranks `G_sub`'s passages.

In [ ]:
def entity_scores_from_beam(beam, G_sub):
    entity_scores = collections.defaultdict(float)

    for path, score in beam:
        for prop_id in path:
            for ent in prop_entities.get(prop_id, []):
                entity_scores[entity_node_id(ent)] += score

        for prop_a, prop_b in zip(path, path[1:]):
            ents_a = prop_entities.get(prop_a, [])
            ents_b = prop_entities.get(prop_b, [])
            linked = any(
                G_sub.has_edge(entity_node_id(ea), entity_node_id(eb))
                and G_sub[entity_node_id(ea)][entity_node_id(eb)].get("synonymy_weight", 0.0) > 0
                for ea in ents_a
                for eb in ents_b
                if entity_node_id(ea) in G_sub
            )
            if linked:
                for ent in set(ents_a) | set(ents_b):
                    entity_scores[entity_node_id(ent)] += SYNONYMY_BRIDGE_BOOST * score

    return entity_scores


def combine_scores(entity_scores, s_initial):
    """Sfinal = entity_scores (beam paths) + Sinitial (stage-1 seed), summed."""
    s_final = collections.defaultdict(float)
    for n, w in entity_scores.items():
        s_final[n] += w
    for n, w in s_initial.items():
        s_final[n] += w
    return s_final


def final_ppr_ranking(G_sub, s_final):
    personalization = normalized_personalization(s_final, G_sub)
    ppr_scores = nx.pagerank(G_sub, alpha=PPR_ALPHA_STAGE2, personalization=personalization, weight="weight")

    passage_scores = [
        (n, s) for n, s in ppr_scores.items() if G_sub.nodes[n].get("node_type") == "passage"
    ]
    passage_scores.sort(key=lambda kv: -kv[1])
    return passage_scores


print("Stage 2 scoring / final-PPR functions defined.")

## 8. Full per-query pipeline

In [ ]:
def retrieve_for_query(question_text, top_k=TOP_K_RESULTS):
    query_vec = embed_texts([question_text], batch_size=1, is_query=True)[0]

    top_props, s_initial, top_passage_ids, G_sub = stage1_coarse_retrieval(query_vec)

    sims_all = prop_embeddings @ query_vec
    sims_by_prop_id = dict(zip(prop_ids, sims_all.tolist()))

    beam = stage2_beam_search(query_vec, sims_by_prop_id, G_sub)

    entity_scores = entity_scores_from_beam(beam, G_sub)
    s_final = combine_scores(entity_scores, s_initial)

    ranked_passages = final_ppr_ranking(G_sub, s_final)[:top_k]
    retrieved = [
        {"pid": pid, "score": float(score), "text": G.nodes[pid].get("text", "")}
        for pid, score in ranked_passages
    ]

    return {
        "retrieved": retrieved,
        "beam": beam,
        "top_passage_ids_stage1": top_passage_ids,
        "g_sub_size": G_sub.number_of_nodes(),
    }


print("retrieve_for_query defined.")

### 8a. Smoke test on a single query

Sanity-check the full pipeline on one query before committing to the 811-query batch run.

In [ ]:
_sample = queries[0]
_t0 = time.perf_counter()
_result = retrieve_for_query(_sample["question"])
_elapsed = time.perf_counter() - _t0

print(f"Q: {_sample['question']}")
print(f"Stage 1 subgraph size: {_result['g_sub_size']} nodes")
print(f"Beam size: {len(_result['beam'])} paths")
print(f"Took {_elapsed:.2f}s")
print()
print("Top retrieved passages:")
for r in _result["retrieved"][:5]:
    print(f"  {r['score']:.4f}  {r['pid']}  {r['text'][:80]!r}")

## 9. Batch runner over all 811 queries

Checkpoints to `retrieval_results.jsonl` (one line per query, flushed immediately) and is
resumable: query ids already present in that file are skipped on re-run. Per-query failures are
caught, logged to `retrieval_failures.jsonl`, and don't stop the batch.

In [ ]:
def already_completed_ids(path):
    done = set()
    if path.exists():
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    done.add(json.loads(line)["query_id"])
                except (json.JSONDecodeError, KeyError):
                    continue
    return done


done_ids = already_completed_ids(RESULTS_PATH)
if done_ids:
    print(f"Resuming: {len(done_ids)} queries already completed in {RESULTS_PATH}")

pending = [q for q in queries if get_query_id(q) not in done_ids]
print(f"{len(pending)} / {len(queries)} queries remaining")

In [ ]:
query_times = []

with open(RESULTS_PATH, "a") as results_f, open(FAILURES_PATH, "a") as failures_f:
    for item in tqdm(pending):
        qid = get_query_id(item)
        question = item["question"]

        t0 = time.perf_counter()
        try:
            result = retrieve_for_query(question)
        except Exception as e:
            failures_f.write(json.dumps({"query_id": qid, "question": question, "error": str(e)}) + "\n")
            failures_f.flush()
            continue

        query_times.append(time.perf_counter() - t0)
        record = {"query_id": qid, "question": question, "retrieved": result["retrieved"]}
        results_f.write(json.dumps(record) + "\n")
        results_f.flush()

        if len(query_times) == PROGRESS_PRINT_AFTER:
            avg = sum(query_times) / len(query_times)
            remaining = len(pending) - len(query_times)
            eta_min = avg * remaining / 60
            print(f"\nAfter {PROGRESS_PRINT_AFTER} queries: avg {avg:.2f}s/query, "
                  f"rough ETA for remaining {remaining} queries: {eta_min:.1f} min")

print(f"Done. {len(query_times)} queries processed successfully this run "
      f"({len(pending) - len(query_times)} failed -- see {FAILURES_PATH}).")

## 10. Evaluation — recall@K (optional sanity check)

For each query, checks whether the gold `supporting_facts` titles are covered by the retrieved
passages' titles within the top-K, for a few values of K.

In [ ]:
def passage_title_from_pid(pid):
    text = G.nodes[pid].get("text", "")
    return text.split("\n", 1)[0]


pid_to_title = {
    n: passage_title_from_pid(n) for n, d in G.nodes(data=True) if d.get("node_type") == "passage"
}

results_by_qid = {}
with open(RESULTS_PATH) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        results_by_qid[rec["query_id"]] = rec

recall_hits = {k: 0 for k in RECALL_AT_K_VALUES}
evaluated = 0

for item in queries:
    qid = get_query_id(item)
    rec = results_by_qid.get(qid)
    if rec is None:
        continue
    gold_titles = {title for title, _ in item.get("supporting_facts", [])}
    if not gold_titles:
        continue
    evaluated += 1

    retrieved_titles_in_order = [pid_to_title.get(r["pid"], "") for r in rec["retrieved"]]
    for k in RECALL_AT_K_VALUES:
        topk_titles = set(retrieved_titles_in_order[:k])
        if gold_titles.issubset(topk_titles):
            recall_hits[k] += 1

print(f"Evaluated {evaluated} queries with results + gold supporting facts")
for k in RECALL_AT_K_VALUES:
    recall = recall_hits[k] / evaluated if evaluated else 0.0
    print(f"recall@{k} (all gold titles covered): {recall:.3f}  ({recall_hits[k]}/{evaluated})")